# Modelling workbench

Notebook separat per provar el pipeline de clustering sense tocar `modelling.ipynb`.

Flux recomanat:
1. Seleccionar dataset d'entrada
2. Validar que no hi hagi `NaN` ni `inf`
3. Escalar amb `StandardScaler`
4. Fer PCA com a diagnostic
5. Comparar clustering amb i sense PCA
6. Revisar metriques i interpretabilitat


## Llibreries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)


## Carrega de dades

In [ ]:
BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / 'data' / 'modelling'

datasets = {
    '2015': pd.read_csv(DATA_DIR / 'df_2015.csv'),
    '2023': pd.read_csv(DATA_DIR / 'df_2023.csv'),
    'deltes': pd.read_csv(DATA_DIR / 'df_deltes.csv'),
}

for name, df in datasets.items():
    print(name, df.shape)


## Seleccio del dataset

Canvia `dataset_name` segons el cas que vulguis provar.
Comenca per `2023` si vols el cas mes directe.

In [ ]:
dataset_name = '2023'

df = datasets[dataset_name].copy()
df.head()


## Validacio minima abans de modelar

In [ ]:
print('Duplicats codi_barri:', df['codi_barri'].duplicated().sum())
print('NaN totals:', int(df.isna().sum().sum()))
print('inf totals:', int(np.isinf(df.select_dtypes(include=[np.number])).sum().sum()))

display(df.isna().sum())


## Matriu de features

Elimina `codi_barri` abans del clustering.

In [ ]:
id_col = 'codi_barri'
feature_cols = [col for col in df.columns if col != id_col]

X = df[feature_cols].copy()
X.shape


## StandardScaler

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols, index=df[id_col])
X_scaled_df.head()


## Correlacions

Serveix per detectar redundancies entre variables abans de clusteritzar.

In [ ]:
corr = X.corr(numeric_only=True)

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title(f'Matriu de correlacio - {dataset_name}')
plt.tight_layout()
plt.show()


## PCA com a diagnostic

Primer mira la variancia explicada abans de decidir si clusteritzes amb PCA.

In [ ]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained = pd.DataFrame({
    'component': np.arange(1, len(feature_cols) + 1),
    'explained_variance_ratio': pca_full.explained_variance_ratio_,
})
explained['cumulative_variance'] = explained['explained_variance_ratio'].cumsum()
explained


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(explained['component'], explained['cumulative_variance'], marker='o')
plt.axhline(0.8, color='orange', linestyle='--', label='80%')
plt.axhline(0.9, color='green', linestyle='--', label='90%')
plt.xlabel('Nombre de components')
plt.ylabel('Variancia acumulada')
plt.title(f'PCA - variancia explicada acumulada ({dataset_name})')
plt.legend()
plt.tight_layout()
plt.show()


## Components PCA per al clustering

Ajusta `n_components` segons la corba anterior.

In [ ]:
n_components = 4

pca = PCA(n_components=n_components, random_state=42)
X_pca = pca.fit_transform(X_scaled)
X_pca_df = pd.DataFrame(X_pca, columns=[f'PC{i}' for i in range(1, n_components + 1)], index=df[id_col])
X_pca_df.head()


## Funcions d'avaluacio

In [ ]:
def evaluate_partition(X_matrix, labels, model_name, representation_name):
    unique_labels = set(labels)
    n_clusters = len(unique_labels - {-1})

    result = {
        'model': model_name,
        'representation': representation_name,
        'n_clusters': n_clusters,
        'has_noise': -1 in unique_labels,
    }

    valid_mask = labels != -1
    valid_labels = labels[valid_mask]

    if len(set(valid_labels)) >= 2:
        X_valid = X_matrix[valid_mask]
        result['silhouette'] = silhouette_score(X_valid, valid_labels)
        result['calinski_harabasz'] = calinski_harabasz_score(X_valid, valid_labels)
        result['davies_bouldin'] = davies_bouldin_score(X_valid, valid_labels)
    else:
        result['silhouette'] = np.nan
        result['calinski_harabasz'] = np.nan
        result['davies_bouldin'] = np.nan

    return result


## KMeans: comparar amb i sense PCA

Comenca provant diferents valors de `k`.

In [ ]:
k_values = range(2, 7)
results = []

for k in k_values:
    km_scaled = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels_scaled = km_scaled.fit_predict(X_scaled)
    results.append(evaluate_partition(X_scaled, labels_scaled, f'KMeans_k{k}', 'scaled'))

    km_pca = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels_pca = km_pca.fit_predict(X_pca)
    results.append(evaluate_partition(X_pca, labels_pca, f'KMeans_k{k}', 'pca'))

results_df = pd.DataFrame(results)
results_df.sort_values(['model', 'representation'])


## Agglomerative: comparar amb i sense PCA

In [ ]:
agg_results = []

for k in k_values:
    agg_scaled = AgglomerativeClustering(n_clusters=k)
    labels_scaled = agg_scaled.fit_predict(X_scaled)
    agg_results.append(evaluate_partition(X_scaled, labels_scaled, f'Agglomerative_k{k}', 'scaled'))

    agg_pca = AgglomerativeClustering(n_clusters=k)
    labels_pca = agg_pca.fit_predict(X_pca)
    agg_results.append(evaluate_partition(X_pca, labels_pca, f'Agglomerative_k{k}', 'pca'))

agg_results_df = pd.DataFrame(agg_results)
agg_results_df.sort_values(['model', 'representation'])


## DBSCAN opcional

Nomes te sentit si vols explorar soroll i formes no esferiques.
No cal fer-lo si KMeans i Agglomerative ja et donen una solucio clara.

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=4)
dbscan_labels = dbscan.fit_predict(X_scaled)

dbscan_result = evaluate_partition(X_scaled, dbscan_labels, 'DBSCAN', 'scaled')
pd.DataFrame([dbscan_result])


## Escollir una solucio candidata

Quan triis model i representacio, guarda les etiquetes i revisa perfils per cluster.

In [ ]:
chosen_k = 3
chosen_representation = 'scaled'  # 'scaled' o 'pca'

if chosen_representation == 'scaled':
    final_model = KMeans(n_clusters=chosen_k, random_state=42, n_init=20)
    final_labels = final_model.fit_predict(X_scaled)
else:
    final_model = KMeans(n_clusters=chosen_k, random_state=42, n_init=20)
    final_labels = final_model.fit_predict(X_pca)

clustered = df[[id_col]].copy()
clustered['cluster'] = final_labels
clustered.head()


In [ ]:
perfil_clusters = df.copy()
perfil_clusters['cluster'] = final_labels
perfil_clusters.groupby('cluster')[feature_cols].mean().T


## Notes finals

- Si la solucio amb PCA millora poc pero perd interpretabilitat, queda't amb les variables originals escalades.
- Si hi ha molta correlacio i PCA simplifica clarament l'estructura, pot tenir sentit clusteritzar sobre PCA.
- No cal provar molts algoritmes al principi: `KMeans` i `Agglomerative` solen ser suficients per una primera comparacio seriosa.
